<a href="https://colab.research.google.com/github/keshavarzIT/AI-Agenting/blob/main/storeAsistantOrder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"An end-to-end Python pipeline using google-genai and Gemini 3.6 Flash to parse unstructured user input and convert it into structured JSON data for automated order processing."

In [12]:
# !pip install python-telegram-bot --upgrade

In [13]:
# !pip install nest_asyncio python-telegram-bot --upgrade

In [14]:
import json
import logging
from google import genai
import nest_asyncio
from google.colab import userdata
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    ContextTypes,
    MessageHandler,
    filters,
)

In [16]:
import json
import logging
import nest_asyncio
from google import genai
from google.colab import userdata
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    ContextTypes,
    ConversationHandler,
    MessageHandler,
    filters,
)

# فعال‌سازی پشتیبانی از Event Loop در گوگل کولب
nest_asyncio.apply()

# ۱. تنظیمات اولیه و خواندن کلیدها از Secrets کولب
GEMINI_KEY = userdata.get("store_api")
TELEGRAM_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")

ai_client = genai.Client(api_key=GEMINI_KEY)

# تعریف وضعیت‌های مکالمه (States)
WAITING_FOR_ADDRESS = 1

SYSTEM_PROMPT = """
تو یک اپراتور ثبت سفارش آنلاین‌شاپ هستی.
متن کاربر را بخوان و اطلاعات سفارش را دقیقاً و فقط در قالب یک JSON معتبر استخراج کن.
هیچ متن اضافه، سلام یا توضیحی ننویس! فقط فرمت JSON زیر را خروجی بده:
{
  "order_intent": true/false,
  "items": [{"name": "نام محصول", "count": تعداد}],
  "address": "آدرس کاربر یا null"
}
"""


# ۲. تابع پاسخ به دستور /start یا cancel
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    # پاک کردن حافظه موقت سفارش قبلی
    context.user_data.clear()
    await update.message.reply_text(
        "سلام! من ربات ثبت سفارش هوشمند هستم. لطفاً سفارش خودتون رو بنویسید (مثلاً: دوتا پیتزا مخلوط میخوام)."
    )
    return ConversationHandler.END


# ۳. پردازش پیام اولیه کاربر (ثبت سفارش)
async def handle_order(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_text = update.message.text
    await update.message.reply_text("⏳ در حال بررسی سفارش شما...")

    try:
        # ارسال به Gemini برای استخراج ساختاریافته سفارش
        ai_response = ai_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=f"{SYSTEM_PROMPT}\n\nپیام کاربر: {user_text}",
        )

        data = json.loads(ai_response.text)

        if data.get("order_intent") and data.get("items"):
            # ذخیره اطلاعات سفارش در حافظه موقت کاربر (context.user_data)
            context.user_data["items"] = data["items"]

            # بررسی اینکه آیا کاربر همراه سفارش، آدرس را هم فرستاده یا خیر
            address = data.get("address")

            if address and address.lower() != "null":
                # اگر آدرس همان اول وجود داشت، فاکتور نهایی صادر می‌شود
                context.user_data["address"] = address
                return await finalize_order(update, context)
            else:
                # اگر آدرس نبود، وارد مرحله دریافت آدرس می‌شویم
                await update.message.reply_text(
                    "📍 لطفاً آدرس دقیق پستی خودتون رو جهت ارسال سفارش وارد کنید:"
                )
                return WAITING_FOR_ADDRESS
        else:
            await update.message.reply_text(
                "❌ متوجه سفارش نشدم. لطفاً نام محصول و تعداد را دقیق‌تر بنویسید."
            )
            return ConversationHandler.END

    except Exception as e:
        await update.message.reply_text(
            "⚠️ خطایی در پردازش رخ داد. لطفاً دوباره تلاش کنید."
        )
        return ConversationHandler.END


# ۴. دریافت آدرس در مرحله بعد
async def handle_address(update: Update, context: ContextTypes.DEFAULT_TYPE):
    address_text = update.message.text
    # ذخیره آدرس در حافظه موقت
    context.user_data["address"] = address_text

    # صدور فاکتور نهایی
    return await finalize_order(update, context)


# ۵. تابع صدور فاکتور نهایی
async def finalize_order(update: Update, context: ContextTypes.DEFAULT_TYPE):
    items = context.user_data.get("items", [])
    address = context.user_data.get("address", "ثبت نشده")

    item_name = items[0]["name"]
    count = items[0]["count"]

    unit_price = 250000
    total_price = count * unit_price

    reply_msg = (
        f"✅ **سفارش شما با موفقیت ثبت شد!**\n\n"
        f"📦 **محصول:** {count} عدد {item_name}\n"
        f"📍 **آدرس ارسال:** {address}\n"
        f"💰 **مبلغ قابل پرداخت:** {total_price:,} تومان\n\n"
        f"برای ثبت سفارش جدید، دوباره پیام بفرستید."
    )

    await update.message.reply_text(reply_msg, parse_mode="Markdown")

    # پاکسازی حافظه موقت کاربر
    context.user_data.clear()
    return ConversationHandler.END


# ۶. راه اندازی ربات با ConversationHandler
if __name__ == "__main__":
    app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()

    # تعریف مدیریت جریان گفتگو
    conv_handler = ConversationHandler(
        entry_points=[
            MessageHandler(filters.TEXT & ~filters.COMMAND, handle_order)
        ],
        states={
            WAITING_FOR_ADDRESS: [
                MessageHandler(
                    filters.TEXT & ~filters.COMMAND, handle_address
                )
            ],
        },
        fallbacks=[CommandHandler("start", start)],
    )

    app.add_handler(CommandHandler("start", start))
    app.add_handler(conv_handler)

    print("🚀 ربات هوشمند آماده دریافت سفارشات دو مرحله‌ای است...")
    app.run_polling()

🚀 ربات هوشمند آماده دریافت سفارشات دو مرحله‌ای است...


RuntimeError: Cannot close a running event loop